<a href="https://colab.research.google.com/github/ThumulaRumesh/Smart-Homes---Indoor-Scene-Recognition-from-Images---CNN/blob/develop/02_Data_Splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Smart Home Indoor Scene Recognition from Images
## Notebook 02: Dataset Splitting

### Purpose

This notebook prepares the selected indoor scene images for model development and evaluation.

The five selected classes from Notebook 01 are divided into training, validation and test datasets. The split is performed before model training so that the models can be trained and evaluated using separate data.

### Selected Classes

- Kitchen
- Bedroom
- Bar
- Living Room
- Warehouse

### Dataset Splits

The dataset is organised into three subsets:

- **Training set** – used to train the classification models.
- **Validation set** – used during training to monitor model performance and apply early stopping/checkpointing.
- **Test set** – kept separate for final evaluation.

The validation and test images are kept unchanged. The training data is later balanced through preprocessing and augmentation in Notebook 03.

### Main Objectives

1. Load the selected classes.
2. Create the required training, validation and test folders.
3. Separate the images into the appropriate datasets.
4. Check the number of images in each class and split.
5. Verify that the dataset structure is suitable for TensorFlow image loading.
6. Preserve the test data for final model evaluation.

This notebook establishes the dataset structure used consistently by all three classification models.

# Step 1 — Set the Paths and Classes

The original dataset path, the five selected classes and the output dataset directory are defined.

A fixed random seed of 42 is used so that the dataset split can be reproduced consistently.

In [1]:
import os
import random
import shutil #to work with files and folders
from sklearn.model_selection import train_test_split

# Original dataset
source_dir = "/content/drive/MyDrive/Colab Notebooks/Y 02 S 02, Advance ML/Assignment/MIT Indoor Scenes/indoorCVPR_09/Images"

# Selected classes
classes = [
    "kitchen",
    "bedroom",
    "bar",
    "livingroom",
    "warehouse"
]

# New dataset location
output_dir = "/content/drive/MyDrive/Colab Notebooks/Y 02 S 02, Advance ML/Assignment/Selected Classes"

# Reproducibility
random.seed(42) #a fixed seed to reproduce the same random split

## Step 2 — Create the Folders

A clean output directory is created for the new train, validation and test datasets.

Any previous generated split is removed before the folders are recreated. This prevents stale files from an earlier experiment from remaining in the dataset.

Each split contains one folder for every selected class, making the structure compatible with TensorFlow's directory-based image loading.

In [2]:
# Remove any previous split so that the dataset is recreated cleanly

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)


splits = [
    "train",
    "validation",
    "test"
]


for split in splits:

    for class_name in classes:

        os.makedirs(
            os.path.join(
                output_dir,
                split,
                class_name
            ),
            exist_ok=True
        )


print("Clean dataset folders created successfully.")

Clean dataset folders created successfully.


In [3]:
print("Source exists:", os.path.exists(source_dir))

for class_name in classes:
    class_path = os.path.join(source_dir, class_name)
    print(class_name, ":", os.path.exists(class_path))

Source exists: True
kitchen : True
bedroom : True
bar : True
livingroom : True
warehouse : True


## Step 3 — Perform the 70/15/15 Split

The images from each selected class are divided into three subsets:

- 70% training
- 15% validation
- 15% testing

The split is performed separately for each class so that all five classes are represented in every dataset subset.

The test images are kept separate so that they can be used later for final evaluation.

In [4]:
for class_name in classes:

    class_path = os.path.join(source_dir, class_name)

    # Get all image files
    images = [
        file for file in os.listdir(class_path)
        if file.lower().endswith((".jpg", ".jpeg", ".png")) #checks whether the filename ends with one of these extensions so only image files are collected
    ]

    # First split:
    # 70% train
    # 30% temporary
    train_images, temp_images = train_test_split(
        images,
        test_size=0.30,
        random_state=42,
        shuffle=True
    )

    # Second split:
    # 15% validation
    # 15% test
    validation_images, test_images = train_test_split(
        temp_images,
        test_size=0.50,
        random_state=42,
        shuffle=True
    )

    # Copy training images
    for image in train_images:
        shutil.copy2(
            os.path.join(class_path, image),
            os.path.join(output_dir, "train", class_name, image)
        )

    # Copy validation images
    for image in validation_images:
        shutil.copy2(
            os.path.join(class_path, image),
            os.path.join(output_dir, "validation", class_name, image)
        )

    # Copy test images
    for image in test_images:
        shutil.copy2(
            os.path.join(class_path, image),
            os.path.join(output_dir, "test", class_name, image)
        )

    print(
        f"{class_name}: "
        f"Train={len(train_images)}, "
        f"Validation={len(validation_images)}, "
        f"Test={len(test_images)}"
    )

kitchen: Train=513, Validation=110, Test=111
bedroom: Train=463, Validation=99, Test=100
bar: Train=422, Validation=91, Test=91
livingroom: Train=494, Validation=106, Test=106
warehouse: Train=354, Validation=76, Test=76


## Step 4 — Verify the Final Split

The number of images in every class and every dataset split is counted.

This verification confirms that the train, validation and test folders were created correctly and that the intended class structure has been maintained.

In [5]:
for split in splits:

    print(f"\n{split.upper()}")

    total = 0

    for class_name in classes:

        class_path = os.path.join(output_dir, split, class_name)

        count = len([
            file for file in os.listdir(class_path)
            if file.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

        total += count

        print(f"{class_name:15} : {count}")

    print(f"Total            : {total}")


TRAIN
kitchen         : 513
bedroom         : 463
bar             : 422
livingroom      : 494
warehouse       : 354
Total            : 2246

VALIDATION
kitchen         : 110
bedroom         : 99
bar             : 91
livingroom      : 106
warehouse       : 76
Total            : 482

TEST
kitchen         : 111
bedroom         : 100
bar             : 91
livingroom      : 106
warehouse       : 76
Total            : 484


### Role of the validation set in the final experiment

The validation partition is kept separate from the training and test sets.

In Notebook 04, this untouched validation partition is used as the pool for stratified cross-validation during hyperparameter selection. After the best configuration is selected, the same validation partition is used during final model training for EarlyStopping and checkpoint selection.

The official test set is kept separate and is used only for the final performance evaluation.